# Augmented training: MLP and CNN (executed run)

Retrains the two winning configurations with training-data augmentation (random shift up to 4 px per axis, rotation up to 15°) and nothing else changed: `configs/augmented.yaml` and `configs/cnn_augmented.yaml`. Targets the distribution-shift weakness, where a 4-pixel shift breaks the original models.

The actual run, outputs intact (except the download links, see the last cell). Run 2026-10-25 on a Tesla T4 with Kaggle's torch 2.11.0 and the project's pinned Lightning 2.6.6 / torchmetrics 1.9.0. To rerun it, follow the steps below and **Run all**.

**Before running**
1. Republish the project code to Kaggle so the dataset contains the augmentation code: locally, `python scripts/publish_kaggle_dataset.py`.
2. Create the notebook from the `mnist-mlp-pipeline` dataset page (**⋮ → New notebook**), then **Add Input** → the `digit-recognizer` competition, **Settings → Accelerator → GPU T4 x2**, **Settings → Internet → On**. Full walkthrough: the intro cell of `kaggle_run_executed.ipynb`.
3. **Run all.** Then download the two checkpoints from the last cell.

The test set is *not* evaluated here: that happens once, locally, after download, with the same code path as the original models, so before/after numbers come from identical evaluation code.

In [1]:
# Same Lightning/torchmetrics versions as the project's requirements.txt; torch is Kaggle's own GPU build.
!pip install -q pytorch-lightning==2.6.6 torchmetrics==1.9.0 python-dotenv==1.2.4

import torch, pytorch_lightning, torchmetrics
print('torch', torch.__version__, '| lightning', pytorch_lightning.__version__, '| torchmetrics', torchmetrics.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE -- enable the GPU in Settings')

torch 2.11.0+cu128 | lightning 2.6.6 | torchmetrics 1.9.0
GPU: Tesla T4


In [2]:
import sys

DATASET_DIR = '/kaggle/input/datasets/ivannamor/mnist-mlp-pipeline'
DATA_DIR = '/kaggle/input/competitions/digit-recognizer'
sys.path.insert(0, DATASET_DIR)

try:
    from src.data_module import build_train_transform  # noqa: F401
except ImportError:
    raise RuntimeError('The Kaggle dataset has old code without augmentation: run scripts/publish_kaggle_dataset.py locally, then attach the new dataset version.') from None
print('project code ready (with augmentation)')

project code ready (with augmentation)


## Train both augmented models

Same settings as the original runs (up to 30 epochs, early stopping on validation loss, best epoch checkpointed), only augmentation added.

In [3]:
import shutil
from src.config import ExperimentConfig
from src.train import train

checkpoints = {}
for name in ['augmented', 'cnn_augmented']:
    config = ExperimentConfig.from_yaml(f'{DATASET_DIR}/configs/{name}.yaml')
    config.data.data_dir = DATA_DIR
    print(f'=== {name}: shift {config.data.augment_max_shift_px}px, rotation {config.data.augment_max_rotation_deg} deg ===')
    trainer = train(config)
    destination = f'/kaggle/working/{name}.ckpt'
    shutil.copy(trainer.checkpoint_callback.best_model_path, destination)
    checkpoints[name] = destination
    print(f'{name}: best val_accuracy {trainer.checkpoint_callback.best_model_score:.4f}, epochs run {trainer.current_epoch}')

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


=== augmented: shift 4px, rotation 15 deg ===


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                  ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net                   │ Sequential                │  535 K │ train │     0 │
│ 1 │ loss_fn               │ CrossEntropyLoss          │      0 │ train │     0 │
│ 2 │ val_accuracy          │ MulticlassAccuracy        │      0 │ train │     0 │
│ 3 │ val_precision         │ MulticlassPrecision       │      0 │ train │     0 │
│ 4 │ val_recall            │ MulticlassRecall          │      0 │ train │     0 │
│ 5 │ test_accuracy         │ MulticlassAccuracy        │      0 │ train │     0 │
│ 6 │ test_precision        │ MulticlassPrecision       │      0 │ train │     0 │
│ 7 │ test_recall           │ MulticlassRecall          │      0 │ train │     0 │
│ 8 │ test_confusion_matrix │ MulticlassConfusionMatrix │      0 │ train │     0 │
└───┴───────────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 535 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 535 K                                                                                                
Total estimated model params size (MB): 2.143                                                                      
Modules in train mode: 16                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                  ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net                   │ Sequential                │  1.7 M │ train │     0 │
│ 1 │ loss_fn               │ CrossEntropyLoss          │      0 │ train │     0 │
│ 2 │ val_accuracy          │ MulticlassAccuracy        │      0 │ train │     0 │
│ 3 │ val_precision         │ MulticlassPrecision       │      0 │ train │     0 │
│ 4 │ val_recall            │ MulticlassRecall          │      0 │ train │     0 │
│ 5 │ test_accuracy         │ MulticlassAccuracy        │      0 │ train │     0 │
│ 6 │ test_precision        │ MulticlassPrecision       │      0 │ train │     0 │
│ 7 │ test_recall           │ MulticlassRecall          │      0 │ train │     0 │
│ 8 │ test_confusion_matrix │ MulticlassConfusionMatrix │      0 │ train │     0 │
└───┴───────────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 1.7 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.7 M                                                                                                
Total estimated model params size (MB): 6.732                                                                      
Modules in train mode: 20                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=30` reached.


Best val_accuracy: 0.9921684861183167
Best checkpoint: outputs/logs/cnn_augmented/version_0/checkpoints/best.ckpt
cnn_augmented: best val_accuracy 0.9922, epochs run 30


## Download the checkpoints

Save both files into the project's `results/` folder locally (`results/augmented.ckpt`, `results/cnn_augmented.ckpt`).

In [4]:
import base64
from IPython.display import HTML, display

def make_download_link(path):
    filename = path.split('/')[-1]
    with open(path, 'rb') as f:
        data = base64.b64encode(f.read()).decode()
    return HTML(f'<a download="{filename}" href="data:application/octet-stream;base64,{data}">Download {filename}</a>')

for path in checkpoints.values():
    display(make_download_link(path))

[Output removed before committing: the links embedded both checkpoints (~35 MB).
 The downloaded files are results/augmented.ckpt and results/cnn_augmented.ckpt.]
